# News Reaction Model v2 Training Metrics

Plots regression learning curves and final 2026 per-horizon RMSE.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

train_root = Path(r'D:/TradingML/runtimes/news-reaction-model/v2/train')
run_dir = max((path for path in train_root.rglob('*') if path.is_dir() and (path / 'metrics.jsonl').exists()), key=lambda path: path.stat().st_mtime)
print('run:', run_dir)
rows = [json.loads(line) for line in (run_dir / 'metrics.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
frame = pd.json_normalize(rows)
display(frame.tail())
sample_column = next((name for name in ('sample', 'step', '_step') if name in frame), None)
if sample_column:
    columns = [name for name in ('train/mse', 'val/mse', 'val/mae', 'train/learning_rate') if name in frame]
    frame.plot(x=sample_column, y=columns, subplots=True, figsize=(12, 10), grid=True)
    plt.tight_layout()
final = frame.iloc[-1]
per_horizon = {key: value for key, value in final.items() if key.startswith('val/') and key.endswith('/rmse') and key.count('/') == 2}
pd.Series(per_horizon).sort_values().plot.barh(figsize=(10, 6), title='2026 RMSE by horizon')
